# 지시를 따르도록 미세 튜닝하기 목차
* [Chapter 1 지시 미세 튜닝 소개](#chapter1)
* [Chapter 2 지도 학습 지시 미세 튜닝을 위해 데이터셋 준비하기](#chapter2)

## Chapter 1 지시 미세 튜닝 소개 <a class="anchor" id="chapter1"></a>
1. LLM을 사전 훈련하는 것은 한번에 한 단어씩 생성하는 방법을 배우넌 것입니다.
   - 이렇게 만들어진 사전 훈련된 LLM은 텍스트 완성능력이 있다.
   - "이 텍스트 문법을 고쳐 줘"와 같은 구체적인 명령을 잘 수행하지 못합니다.

2. 지시를 따르고 기대하는 응답을 생성하도록 LLM 능력을 향상시켜봅시다.

    ![예시](image/07-01-example5.png)

3. 지시 미세튜닝은 세 단계 과정을 거친다.

   ![순서](image/07-01-process2.png)

## Chapter 2 지도 학습 지시 미세 튜닝을 위해 데이터셋 준비하기 <a class="anchor" id="chapter2"></a>
1. 지시 미세 튜닝을 위해서는 지시와 그에 대한 응답이 포함된 데이터셋이 필요하다.
   - 예: "이 텍스트 문법을 고쳐 줘" -> "고쳐진 텍스트"

In [4]:
# 데이터셋 다운로드
import json
import os
import urllib

def download_and_load_file(file_path, url):
    if not os.path.exists(file_path):
        with urllib.request.urlopen(url) as response:
            text_data = response.read().decode('utf-8')
        with open(file_path,'w', encoding='utf-8') as file:
            file.write(text_data)
    with open(file_path, 'r', encoding='utf-8') as file:
        data = json.load(file)
    return data

file_path = "instruction-data.json"
url = (
    "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch"
    "/main/ch07/01_main-chapter-code/instruction-data.json"
)

data = download_and_load_file(file_path, url)
print("샘플 개수:", len(data))


샘플 개수: 1100


2. 샘플 데이터 양식은 'instruction', 'input', 'output' 키로 구성된 파이썬 딕셔너리의 리스트이다.
    - 'instruction': 모델이 수행해야 할 작업 지시
    - 'input': 작업에 필요한 추가 정보(없을 수도 있음)
    - 'output': 모델이 생성해야 할 기대 응답

In [ ]:
# 샘플 데이터 출력
#   - 'instruction', 'input', 'output' 키로 구성된 파이썬 딕셔너리의 리스트
print("샘플 데이터 예시:\n", data[50])

# input 필드가 비어있는 경우도 있음
#   - 
print("다른 샘플 데이터 예시:\n", data[999])

샘플 데이터 예시:
 {'instruction': 'Identify the correct spelling of the following word.', 'input': 'Ocassion', 'output': "The correct spelling is 'Occasion.'"}
다른 샘플 데이터 예시:
 {'instruction': "What is an antonym of 'complicated'?", 'input': '', 'output': "An antonym of 'complicated' is 'simple'."}



3. input 필드가 비어있는 경우도 있음
   - 예: "이 텍스트 문법을 고쳐 줘" -> "고쳐진 텍스트"

4. LLM을 위해 샘플을 포멧팅하는 방법은 여러가지가 있다.
   - 알파카(Alpaca) 스타일 포멧
      - 지시, 입력, 응답 섹션으로 구성된다.
   - Phi-3 스타일 포멧
      - <|user|>와 <|assistant|> 토큰으로 구성된 간단한 포멧을 사용한다.
   - 이른 종종 프롬프트 스타일이라고 부른다.

      ![스타일](image/07-01-style2.png)

5. 알파카는 초기 LLM 중 하나로 지시 미세 튜닝 과정에 대한 내용이 공개되어 있다.
   - 알파카 스타일 포멧이 널리 사용된다.
   - 본 노트북에서는 알파카 스타일 포멧을 사용한다.

In [9]:
# data 리스트의 항목을 알파카 스타일 포맷으로 변환
def format_input(entry):
    instruction_text = (
        f"Below is an instruction that describes a task. "
        f"Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )

    input_text = f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""

    return instruction_text + input_text

In [10]:
model_input = format_input(data[50])
desired_response = f"\n\n### Response:\n{data[50]['output']}"
print(model_input+desired_response)

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Identify the correct spelling of the following word.

### Input:
Ocassion

### Response:
The correct spelling is 'Occasion.'


In [11]:
# format_input 함수는 값이 비어 있을 때 input 섹션을 건너뜁니다.
model_input_no_input = format_input(data[999])
desired_response_no_input = f"\n\n### Response:\n{data[999]['output']}"
print(model_input_no_input+desired_response_no_input)

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
What is an antonym of 'complicated'?

### Response:
An antonym of 'complicated' is 'simple'.


6. 데이터 셋츨 훈련 세트, 검증 세트, 테스트 세트로 나눈다.

In [ ]:
train_portion = int(len(data) * 0.85)
val_portion = int(len(data) * 0.1)
test_portion = len(data) - train_portion - val_portion

train_data = data[:train_portion]
val_data = data[train_portion:train_portion + val_portion]
test_data = data[train_portion + val_portion:]

print("훈련 세트 크기:", len(train_data))
print("테스트 세트 크기:", len(test_data))
print("검증 세트 크기:", len(val_data))

훈련 세트 크기: 935
검증 세트 크기: 110
테스트 세트 크기: 55
